In [4]:
import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt
import io
from datetime import datetime

# Load model
model = tf.keras.models.load_model('Dronedetectionsystem.h5')
class_names = ['Airplane', 'Birds', 'Drone']

# Store prediction history
prediction_history = []

def predict_image(image):
    """Predict image class"""
    try:
        img_array = np.array(image)
        resized = tf.image.resize(img_array, (256, 256))
        scaled = resized / 255.0
        batch = np.expand_dims(scaled, axis=0)
        
        yhat = model.predict(batch, verbose=0)
        predictions = {
            class_names[i]: float(yhat[0][i])
            for i in range(len(class_names))
        }
        return predictions
    except Exception as e:
        return {"Error": str(e)}


def create_probability_chart(yhat):
    """Create a bar chart of class probabilities and return as PIL image"""
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#FF6B6B' if i == np.argmax(yhat[0]) else '#4ECDC4' for i in range(len(class_names))]
    bars = ax.barh(class_names, yhat[0] * 100, color=colors, edgecolor='black', linewidth=1.2)
    ax.set_xlim(0, 100)
    ax.set_xlabel('Probability (%)', fontsize=12)
    ax.set_title('Class Probability Distribution', fontsize=14, fontweight='bold')
    
    for bar, prob in zip(bars, yhat[0] * 100):
        ax.text(prob + 2, bar.get_y() + bar.get_height() / 2, f'{prob:.1f}%',
                va='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    plt.close()
    buf.seek(0)
    return Image.open(buf)


def get_image_metadata(image):
    """Extract image metadata"""
    try:
        img_array = np.array(image)
        return {
            'dimensions': f"{img_array.shape[1]} x {img_array.shape[0]} pixels",
            'channels': img_array.shape[2] if len(img_array.shape) > 2 else 1,
            'data_type': str(img_array.dtype),
            'size_kb': (img_array.nbytes / 1024)
        }
    except:
        return {'dimensions': 'Unknown', 'channels': 'Unknown', 'data_type': 'Unknown', 'size_kb': 0}


def predict_with_details(image):
    """Predict with detailed information"""
    try:
        img_array = np.array(image)
        resized = tf.image.resize(img_array, (256, 256))
        scaled = resized / 255.0
        batch = np.expand_dims(scaled, axis=0)
        
        yhat = model.predict(batch, verbose=0)
        predicted_class = np.argmax(yhat)
        confidence = float(np.max(yhat))
        
        top_2_indices = np.argsort(yhat[0])[-2:][::-1]
        top_2_classes = [class_names[i] for i in top_2_indices]
        top_2_probs = [float(yhat[0][i]) * 100 for i in top_2_indices]
        
        metadata = get_image_metadata(image)
        
        output_text = f"""
🎯 PREDICTION RESULTS
═══════════════════════════════════

📌 Predicted Class: {class_names[predicted_class]}
📊 Confidence: {confidence * 100:.2f}%

📋 CLASS PROBABILITIES:
"""
        
        for i, class_name in enumerate(class_names):
            prob = float(yhat[0][i]) * 100
            bar = "█" * int(prob / 5) + "░" * (20 - int(prob / 5))
            output_text += f"\n   {class_name}: {bar} {prob:.2f}%"
        
        output_text += f"""

🏆 TOP 2 PREDICTIONS:
   1st: {top_2_classes[0]} ({top_2_probs[0]:.2f}%)
   2nd: {top_2_classes[1]} ({top_2_probs[1]:.2f}%)

📷 IMAGE METADATA:
   Dimensions: {metadata['dimensions']}
   Channels: {metadata['channels']}
   Data Type: {metadata['data_type']}
   Size: ~{metadata['size_kb']:.1f} KB

⏰ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
═══════════════════════════════════
"""
        
        chart_image = create_probability_chart(yhat)
        
        prediction_history.append({
            'timestamp': datetime.now().strftime('%H:%M:%S'),
            'prediction': class_names[predicted_class],
            'confidence': confidence * 100
        })
        if len(prediction_history) > 5:
            prediction_history.pop(0)
        
        return output_text, chart_image
    except Exception as e:
        return f"❌ Error: {str(e)}", None

# Create interface with tabs
with gr.Blocks(theme=gr.themes.Soft(), title="Civillian Drone Detection System") as demo:
    gr.Markdown("""
# 🐦 Civillian Drone Detection System

Upload an image to classify it as a **Bird**, **Drone**, or **Airplane**.

This model uses a deep convolutional neural network trained on thousands of images.
""")
    
    with gr.Tabs():
        with gr.TabItem("🎯 Quick Prediction"):
            with gr.Row():
                with gr.Column():
                    image_input = gr.Image(type="pil", label="Upload Image")
                    submit_btn = gr.Button("🔍 Classify", variant="primary")
                
                with gr.Column():
                    output_label = gr.Label(num_top_classes=3, label="Predictions")
            
            submit_btn.click(
                fn=predict_image,
                inputs=image_input,
                outputs=output_label
            )
        
        with gr.TabItem("📊 Detailed Analysis"):
            with gr.Row():
                with gr.Column():
                    image_input2 = gr.Image(type="pil", label="Upload Image")
                    submit_btn2 = gr.Button("📈 Analyze", variant="primary")
                
                with gr.Column():
                    output_text = gr.Textbox(
                        label="Analysis Results",
                        lines=18,
                        interactive=False
                    )
            
            with gr.Row():
                chart_output = gr.Image(label="Probability Chart", type="pil")
            
            submit_btn2.click(
                fn=predict_with_details,
                inputs=image_input2,
                outputs=[output_text, chart_output]
            )
        
        with gr.TabItem("ℹ️ About"):
            gr.Markdown("""
## Model Information

- **Architecture**: Convolutional Neural Network (CNN)
- **Input Size**: 256 × 256 pixels
- **Classes**: 3 (Bird, Drone, Airplane)
- **Framework**: TensorFlow/Keras

## How to Use

1. Upload an image (JPG, PNG, etc.)
2. Click "Classify" button
3. View the predictions and confidence scores

## Tips for Best Results

- Use clear, well-lit images
- Ensure the subject is clearly visible
- Avoid blurry or heavily cropped images
- Images will be automatically resized to 256×256

---

**Made with ❤️ using TensorFlow and Gradio**
""")

if __name__ == "__main__":
    demo.launch(share=True)

Running on local URL:  http://127.0.0.1:7863


c:\Users\Govind Gupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.39.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
